# Exercícios — Extração de dados via API (Gabarito)

> Esta é a versão **resolvida e comentada** dos exercícios em
> [../01_exercicios_extracao_api.ipynb](../01_exercicios_extracao_api.ipynb). Cada
> linha de código traz um comentário explicando o quê e, principalmente, **por
> quê** — use isso para conferir respostas ou para se apoiar na correção, não
> para substituir a tentativa dos alunos.

Cada célula de código abaixo é uma solução possível — não a única. Se um aluno
resolveu de outro jeito e chegou num resultado correto, isso também vale.

In [ ]:
import calendar  # calcula quantos dias um mês tem (calendar.monthrange) sem "chutar" o número na mão
import json      # serializa/desserializa o payload da API e os arquivos salvos em disco
import time      # usado no time.sleep() entre requisições e no backoff das retentativas
from pathlib import Path  # caminhos de arquivo multiplataforma — mais seguro que colar strings com "/"

import requests  # cliente HTTP usado para consumir a API da Open-Meteo

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"  # endpoint fixo da Historical Weather API
RAW_DIR = Path("../../data/raw")
# duas pastas acima: gabarito/ -> exercicios/ -> raiz do projeto -> data/raw (mesma pasta usada na Parte 1)
RAW_DIR.mkdir(parents=True, exist_ok=True)
# cria a pasta (e as pastas pai, se faltarem) só se ainda não existir; exist_ok evita erro se já existir

VARIAVEIS_HORARIAS = ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m"]
# lista padrão de variáveis, reaproveitada em várias células — evita repetir a mesma string várias vezes


# Mesma função construída na Parte 1 — incluída aqui para o gabarito rodar sozinho,
# sem depender de ter executado o notebook 01 antes.
def buscar_clima_historico(lat: float, lon: float, data_inicio: str, data_fim: str,
                            variaveis=None) -> dict:
    params = {
        "latitude": lat,             # nome exato de parâmetro que a API espera na query string
        "longitude": lon,
        "start_date": data_inicio,   # formato esperado pela API: "AAAA-MM-DD"
        "end_date": data_fim,
        "hourly": ",".join(variaveis or VARIAVEIS_HORARIAS),
        # "variaveis or VARIAVEIS_HORARIAS": se quem chamou não passar uma lista customizada, usa a
        # lista padrão; ",".join(...) porque a API espera as variáveis separadas por vírgula (string),
        # não uma lista Python
        "timezone": "America/Sao_Paulo",  # sem isso os horários viriam em UTC, não no fuso local
    }
    resposta = requests.get(BASE_URL, params=params, timeout=30)
    # timeout=30: evita que o código trave indefinidamente se a API não responder
    resposta.raise_for_status()
    # lança uma exceção se o status HTTP for de erro (4xx/5xx) — falha rápido e com uma mensagem clara,
    # em vez de seguir adiante tratando um payload de erro como se fosse um dado válido
    return resposta.json()
    # .json() já desserializa o corpo da resposta HTTP direto para um dict Python

## 🟢 Exercício 1 — Uma sexta cidade

Escolha uma **6ª cidade brasileira** (que não esteja entre São Paulo, Rio de
Janeiro, Manaus, Porto Alegre e Recife) e busque os dados horários de temperatura,
umidade, precipitação e vento para **janeiro de 2025**, usando a mesma lógica da
Parte 1. Salve o resultado em `data/raw/clima_raw_<slug>.json`, seguindo a mesma
convenção de nome usada em aula.

In [ ]:
CIDADE_EXTRA = {"nome_exibicao": "Belém", "lat": -1.4558, "lon": -48.4902}
# dicionário simples, no mesmo formato usado em CIDADES na Parte 1 — só para guardar os dados da cidade escolhida

payload = buscar_clima_historico(
    CIDADE_EXTRA["lat"], CIDADE_EXTRA["lon"], "2025-01-01", "2025-01-31"
)
# reaproveita a função do setup — mesma lógica e mesmo período (jan/2025) da Parte 1; só muda a cidade

caminho = RAW_DIR / "clima_raw_belem.json"
# Path aceita o operador "/" para juntar pedaços de caminho — mais legível que concatenar strings na mão

with open(caminho, "w", encoding="utf-8") as f:
    # "w" (write) sobrescreve o arquivo se ele já existir; encoding="utf-8" evita problemas com acentos
    json.dump(payload, f, ensure_ascii=False)
    # ensure_ascii=False mantém acentos como caracteres normais no arquivo, em vez de escapá-los (\u00e9)

print(f"Salvo: {caminho}")
# feedback simples para confirmar a gravação sem precisar abrir a pasta manualmente

## 🟢 Exercício 2 — Uma nova variável

Adicione a variável horária `cloud_cover` (cobertura de nuvens, em %) ao pedido
para a cidade escolhida no Exercício 1. Confira, no JSON de resposta, se a nova
variável aparece dentro de `hourly` e qual é a sua unidade (`hourly_units`).

In [ ]:
payload_com_nuvens = buscar_clima_historico(
    CIDADE_EXTRA["lat"], CIDADE_EXTRA["lon"], "2025-01-01", "2025-01-31",
    variaveis=VARIAVEIS_HORARIAS + ["cloud_cover"],
)
# VARIAVEIS_HORARIAS + ["cloud_cover"] concatena listas — evita reescrever as 4 variáveis originais na mão

print("cloud_cover está em hourly?", "cloud_cover" in payload_com_nuvens["hourly"])
# "in" sobre um dict testa se a CHAVE existe — forma direta de confirmar que a API aceitou o parâmetro novo

print("Unidade:", payload_com_nuvens["hourly_units"]["cloud_cover"])
# hourly_units é o dicionário paralelo de metadados (unidade de cada variável), visto na Parte 1

## 🟢 Exercício 3 — Checando consistência

Escreva uma função `verificar_consistencia(payload)` que recebe o dicionário
retornado pela API e devolve `True` se todos os arrays dentro de `hourly` tiverem o
mesmo tamanho, ou `False` caso contrário. Teste com o payload da cidade que você
buscou no Exercício 1.

In [ ]:
def verificar_consistencia(payload: dict) -> bool:
    tamanhos = {len(valores) for valores in payload["hourly"].values()}
    # set-comprehension: guarda o TAMANHO de cada array dentro de hourly; um set descarta duplicatas sozinho
    return len(tamanhos) == 1
    # se todos os arrays tiverem o mesmo tamanho, o set terá só 1 elemento único — é o teste de consistência


print(verificar_consistencia(payload))            # True
# testando com o payload de Belém (sem cloud_cover)
print(verificar_consistencia(payload_com_nuvens))  # True
# testando também com a variável extra — confirma que a função funciona independente de quantas variáveis existam

## 🟢 Exercício 4 — Quantas horas esperar?

Para janeiro de 2025 (31 dias), quantas horas de dado vocês esperam receber?
Confira se o valor bate com o que a API realmente devolveu para a sua cidade. E
para fevereiro de 2025?

⚠️ Escreva o código de forma que funcione para **qualquer mês**, sem "chutar" o
número de dias manualmente (dica: módulo `calendar` da biblioteca padrão, função
`calendar.monthrange`).

In [ ]:
def horas_esperadas(ano: int, mes: int) -> int:
    _, dias_no_mes = calendar.monthrange(ano, mes)
    # monthrange devolve (dia da semana do 1º dia do mês, quantidade de dias do mês) — só o 2º valor
    # importa aqui; "_" é a convenção para dizer "recebo esse valor mas não vou usá-lo"
    return dias_no_mes * 24
    # cada dia do mês gera 24 horas de leitura


print("Janeiro/2025:", horas_esperadas(2025, 1))  # 744
print("Fevereiro/2025:", horas_esperadas(2025, 2))  # 672 (2025 não é bissexto)

print("Horas recebidas de fato:", len(payload["hourly"]["time"]))
# len() de qualquer lista dentro de hourly serve, já que arrays paralelos têm sempre o mesmo tamanho (Exercício 3)

assert len(payload["hourly"]["time"]) == horas_esperadas(2025, 1)
# assert interrompe o notebook com um erro claro se a suposição falhar — uma forma simples de "testar"
# a suposição inline, sem precisar de um framework de testes para um caso pontual como esse

---

## 🔴 Desafio 1 — Períodos longos, em pedaços

Escreva uma função `buscar_periodo_longo(lat, lon, data_inicio, data_fim)` que
funcione mesmo se o intervalo pedido for maior do que a API aceitaria de uma vez
(não é uma limitação real da Open-Meteo, mas simule esse cenário para praticar): a
função deve dividir o intervalo em pedaços de no máximo 31 dias, fazer **uma
chamada por pedaço**, e juntar (concatenar) os arrays de `hourly` de todos os
pedaços em um único dicionário no final.

In [ ]:
from datetime import date, timedelta
# date representa uma data (sem hora); timedelta representa uma duração — usado para "somar dias" a uma data


def buscar_periodo_longo(lat: float, lon: float, data_inicio: str, data_fim: str) -> dict:
    inicio = date.fromisoformat(data_inicio)
    # converte a string "AAAA-MM-DD" num objeto date de verdade, que permite fazer conta de datas
    fim = date.fromisoformat(data_fim)

    hourly_completo = None
    # começa vazio — só descobrimos o "formato" de hourly depois da primeira chamada
    cursor = inicio
    # "cursor" percorre o intervalo pedaço a pedaço, começando pelo início

    while cursor <= fim:
        # continua enquanto ainda houver datas do intervalo pedido não cobertas
        fim_pedaco = min(cursor + timedelta(days=30), fim)
        # cada pedaço tem no máximo 31 dias (30 de diferença + o dia inicial); min(...) garante que o
        # último pedaço não ultrapasse a data final pedida
        pedaco = buscar_clima_historico(lat, lon, cursor.isoformat(), fim_pedaco.isoformat())
        # isoformat() converte o date de volta para string "AAAA-MM-DD", formato que a função espera

        if hourly_completo is None:
            hourly_completo = {chave: list(valores) for chave, valores in pedaco["hourly"].items()}
            # na primeira volta do loop, copia a estrutura inteira; list(valores) cria uma lista própria,
            # em vez de reaproveitar a lista do payload original (evita efeitos colaterais inesperados)
        else:
            for chave, valores in pedaco["hourly"].items():
                hourly_completo[chave].extend(valores)
                # nas voltas seguintes, gruda os novos valores no final de cada lista já existente —
                # extend (não append!) porque estamos juntando LISTAS inteiras, não um item isolado

        cursor = fim_pedaco + timedelta(days=1)
        # avança para o dia seguinte ao fim do pedaço recém-buscado, evitando repetir esse dia na próxima volta

    return {"hourly": hourly_completo}
    # devolve no mesmo "formato" da API original (dict com a chave "hourly"), para quem usar esse
    # resultado poder tratar exatamente como um payload normal


resultado = buscar_periodo_longo(CIDADE_EXTRA["lat"], CIDADE_EXTRA["lon"], "2025-01-01", "2025-03-31")
# pede jan+fev+mar/2025 — mais de 31 dias, então necessariamente passa por mais de uma chamada internamente
print("Total de horas (jan+fev+mar/2025):", len(resultado["hourly"]["time"]))

## 🔴 Desafio 2 — Retentativas com backoff

Adicione tratamento de erro à função de requisição: se a API devolver um erro de
rede (timeout, erro 5xx), tente novamente até 3 vezes, esperando um pouco mais a
cada tentativa (*backoff*), antes de desistir e propagar o erro. Use `try`/`except`
em torno da chamada `requests.get`, capturando exceções específicas — nunca um
`except:` genérico.

In [ ]:
def buscar_com_retentativa(lat: float, lon: float, data_inicio: str, data_fim: str,
                            max_tentativas: int = 3) -> dict:
    for tentativa in range(1, max_tentativas + 1):
        # começa em 1 (não em 0) só para os prints de log ficarem mais legíveis para quem estiver lendo
        try:
            return buscar_clima_historico(lat, lon, data_inicio, data_fim)
            # se a chamada funcionar, o return encerra a função aqui — não passa pelo except nem tenta de novo
        except (requests.Timeout, requests.HTTPError) as erro:
            # captura APENAS erros de rede/HTTP esperados — nunca "except:" genérico, que esconderia
            # até erros de programação (ex. um NameError) como se fossem falha de rede
            if tentativa == max_tentativas:
                raise
                # na última tentativa, desiste e deixa o erro subir para quem chamou a função — sem isso,
                # a função "engoliria" o erro silenciosamente e devolveria None sem avisar ninguém
            espera = tentativa * 2
            # backoff simples: espera mais tempo a cada nova tentativa (2s, depois 4s...), dando mais
            # chance da API se recuperar em vez de martelar ela repetidamente sem pausa
            print(f"Tentativa {tentativa} falhou ({erro}), aguardando {espera}s...")
            time.sleep(espera)


# Teste com uma cidade válida — deve funcionar de primeira, sem cair no except
teste_retentativa = buscar_com_retentativa(CIDADE_EXTRA["lat"], CIDADE_EXTRA["lon"], "2025-01-01", "2025-01-31")
print("Horas recebidas:", len(teste_retentativa["hourly"]["time"]))

## 🔴 Desafio 3 — Menor temperatura, sem pandas

As 5 cidades originais (São Paulo, Rio de Janeiro, Manaus, Porto Alegre, Recife) já
têm arquivos salvos em `data/raw` (gerados na Parte 1). **Sem usar pandas** — só com
`json` e as listas/dicionários do Python puro — escreva um código que descubra qual
das 5 cidades teve a **temperatura mínima mais baixa** em janeiro de 2025, e em que
hora/dia isso aconteceu.

In [ ]:
menor_temp = None
# começa como None (em vez de 0 ou -infinito) para que a PRIMEIRA comparação sempre "vença" e defina o valor inicial
cidade_vencedora = None
hora_vencedora = None

for arquivo in RAW_DIR.glob("clima_raw_*.json"):
    # glob varre a pasta procurando arquivos que batem com esse padrão de nome — exatamente os 5 da Parte 1
    with open(arquivo, encoding="utf-8") as f:
        payload_cidade = json.load(f)
        # json.load lê e já desserializa o conteúdo do ARQUIVO (diferente de json.loads, que recebe uma string)

    for hora, temp in zip(payload_cidade["hourly"]["time"], payload_cidade["hourly"]["temperature_2m"]):
        # zip percorre as duas listas paralelas ao mesmo tempo, par a par — é a versão "sem pandas"
        # de acessar time[i] e temperature_2m[i] juntos, para o mesmo i
        if menor_temp is None or temp < menor_temp:
            # "menor_temp is None" cobre a primeira iteração, quando ainda não há nada para comparar
            menor_temp = temp
            cidade_vencedora = arquivo.stem.replace("clima_raw_", "")
            # .stem pega o nome do arquivo sem a extensão (ex. "clima_raw_manaus"); replace remove o
            # prefixo fixo, sobrando só o slug da cidade
            hora_vencedora = hora

print(f"Cidade com menor temperatura: {cidade_vencedora}")
print(f"Temperatura: {menor_temp}°C, em {hora_vencedora}")

## 🔴 Desafio 4 (livre) — Fora do Brasil

Escolha qualquer cidade **fora do Brasil** e repita a extração da Parte 1 para ela.
O que muda na forma como vocês interpretariam o parâmetro `timezone` nesse caso?
Existe algum cuidado extra que alguém analisando dados de, por exemplo, Tóquio,
precisaria ter (fuso horário, hemisfério, estações do ano invertidas)? Escreva suas
conclusões em uma célula markdown.

**Resposta modelo (exercício aberto — não existe um único "certo"):**

- **Timezone:** o parâmetro `timezone` precisa apontar para o fuso da cidade
  escolhida (ex. `"Asia/Tokyo"` para Tóquio), não `"America/Sao_Paulo"`. Se
  deixássemos o fuso brasileiro, os horários viriam deslocados — a "meia-noite" no
  dado não corresponderia à meia-noite real de lá.
- **Comparar cidades de fusos diferentes:** ao juntar Tóquio com as cidades
  brasileiras num mesmo `DataFrame`, os timestamps tz-naive deixam de ser
  comparáveis diretamente — duas linhas com o mesmo `datetime` não representariam
  o mesmo instante. Seria necessário usar `tz_localize` em cada cidade (com o fuso
  correto) e depois `tz_convert` para um fuso comum (ex. UTC) antes de comparar.
- **Hemisfério/estações:** Tóquio fica no hemisfério norte — enquanto é verão em
  janeiro no Brasil, é **inverno** lá. Uma análise de "temperatura de verão" que
  simplesmente comparasse o mês de janeiro entre as duas cidades estaria, na
  verdade, comparando estações opostas. Para comparar estações equivalentes, seria
  preciso alinhar por estação (ex. verão do hemisfério norte = ~junho–agosto), não
  pelo mesmo mês do calendário.

```python
CIDADE_FORA_BRASIL = {"nome_exibicao": "Tóquio", "lat": 35.6762, "lon": 139.6503}
# lat/lon de Tóquio — mesmo formato usado para as cidades brasileiras

payload_toquio = buscar_clima_historico(
    CIDADE_FORA_BRASIL["lat"], CIDADE_FORA_BRASIL["lon"], "2025-01-01", "2025-01-31"
)
# Nota: a chamada acima ainda usa "America/Sao_Paulo" fixo dentro de
# buscar_clima_historico — o exercício pede exatamente para questionar isso.
# Uma versão mais correta receberia o timezone como parâmetro da função, em vez de
# ter esse valor fixo ("hardcoded") dentro dela.
```